# K-Means Clustering from Scratch — Class Notes

**Unsupervised** algorithm — no labels (y) needed. Finds natural groupings in data.

---

### The 4-step loop

```
1. Pick k random data points as starting centroids
2. Assign every point to its nearest centroid  (by Euclidean distance)
3. Move each centroid to the mean of its assigned points
4. Repeat 2–3 until centroids stop moving  (convergence)
```

| | Supervised (LR, GD) | Unsupervised (K-Means) |
|---|---|---|
| Input | X and y | X only |
| Output | Predicted class / value | Cluster ID per point |
| Learns from | Labels | Distance / structure |
| Method | `fit(X, y)` | `fit_predict(X)` |

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

## Step 1 — Class Setup (`__init__`)

In [ ]:
class KMeans:

    def __init__(self, n_clusters=2, max_iter=100):
        self.n_clusters = n_clusters  # k — how many groups to find
        self.max_iter   = max_iter    # safety cap: stop even if not converged
        self.centroids  = None        # cluster centres, learned during fit_predict()

## Step 2 — `fit_predict` — The Main Loop

Note: K-Means combines fit and predict into one method because cluster labels only make sense after fitting — there's no separate test set in unsupervised learning.

In [ ]:
    def fit_predict(self, X):

        # ── Initialise: pick k random data points as centroids ──────────
        random_index = random.sample(range(0, X.shape[0]), self.n_clusters)
        self.centroids = X[random_index]
        # random.sample → unique indices (no repeats)
        # centroids shape: (n_clusters, n_features)
        # e.g. k=3, 2 features → centroids shape = (3, 2)

        for i in range(self.max_iter):

            # ── Assign: label each point with its nearest centroid ──────
            cluster_group = self.assign_clusters(X)
            # returns array of shape (n_samples,) with values 0,1,...,k-1

            old_centroids = self.centroids

            # ── Move: recompute centroids as mean of assigned points ────
            self.centroids = self.move_centroids(X, cluster_group)

            # ── Convergence check: stop if centroids didn't move ────────
            if (old_centroids == self.centroids).all():
                break
            # .all() → True only if every element is equal
            # centroids stop moving → every point is already in its correct cluster

        return cluster_group   # cluster label per point: e.g. [0, 1, 0, 2, 1, ...]

## Step 3 — `assign_clusters` — Label Every Point

### Euclidean distance via dot product

```
d(a, b) = ||a − b||  =  sqrt( Σ(aᵢ − bᵢ)² )

np.dot(a−b, a−b)  =  Σ(aᵢ−bᵢ)²     (sum of squared differences)
np.sqrt(...)      =  Euclidean distance

Equivalent to: np.linalg.norm(row − centroid)
```

In [ ]:
    def assign_clusters(self, X):
        cluster_group = []
        distances     = []

        for row in X:                          # loop over every data point
            for centroid in self.centroids:    # compute distance to each centroid

                distances.append(
                    np.sqrt(np.dot(row - centroid, row - centroid))
                )
                # np.dot(v, v) = |v|² — faster than np.linalg.norm for 1D vectors

            min_distance = min(distances)
            index_pos    = distances.index(min_distance)
            # index_pos = which centroid was closest → use as cluster label
            # e.g. if centroid 2 was closest → label this point as cluster 2

            cluster_group.append(index_pos)
            distances.clear()   # reset distance list for next data point

        return np.array(cluster_group)   # shape: (n_samples,)

## Step 4 — `move_centroids` — Recompute Cluster Centres

New centroid = **mean position of all points assigned to that cluster**.

```
X[cluster_group == 0]        → all rows in cluster 0
.mean(axis=0)                → column-wise mean → one value per feature
                               = new centroid position for cluster 0
```

In [ ]:
    def move_centroids(self, X, cluster_group):
        new_centroids = []
        cluster_type  = np.unique(cluster_group)   # e.g. [0, 1, 2] for k=3

        for type in cluster_type:
            new_centroids.append(
                X[cluster_group == type].mean(axis=0)
            )
            # X[cluster_group == type] → boolean mask → rows of this cluster
            # .mean(axis=0)            → axis=0 = average over rows
            #                            result shape: (n_features,)

        return np.array(new_centroids)   # shape: (n_clusters, n_features)

## Full Class (clean copy)

In [ ]:
class KMeans:

    def __init__(self, n_clusters=2, max_iter=100):
        self.n_clusters = n_clusters
        self.max_iter   = max_iter
        self.centroids  = None

    def fit_predict(self, X):
        random_index   = random.sample(range(0, X.shape[0]), self.n_clusters)
        self.centroids = X[random_index]

        for i in range(self.max_iter):
            cluster_group = self.assign_clusters(X)
            old_centroids = self.centroids
            self.centroids = self.move_centroids(X, cluster_group)
            if (old_centroids == self.centroids).all():
                break

        return cluster_group

    def assign_clusters(self, X):
        cluster_group = []
        distances = []
        for row in X:
            for centroid in self.centroids:
                distances.append(np.sqrt(np.dot(row - centroid, row - centroid)))
            min_distance = min(distances)
            index_pos    = distances.index(min_distance)
            cluster_group.append(index_pos)
            distances.clear()
        return np.array(cluster_group)

    def move_centroids(self, X, cluster_group):
        new_centroids = []
        cluster_type  = np.unique(cluster_group)
        for type in cluster_type:
            new_centroids.append(X[cluster_group == type].mean(axis=0))
        return np.array(new_centroids)

## Quick Test

In [ ]:
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

km = KMeans(n_clusters=3, max_iter=100)
labels = km.fit_predict(X)

plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='winter', s=80)
plt.scatter(km.centroids[:, 0], km.centroids[:, 1],
            c='red', s=200, marker='X', label='Centroids')
plt.legend()
plt.title('K-Means from scratch — 3 clusters')
plt.show()

## Compare with sklearn

In [ ]:
from sklearn.cluster import KMeans as SKLearnKMeans

sk_km = SKLearnKMeans(n_clusters=3, random_state=42)
sk_labels = sk_km.fit_predict(X)

plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], c=sk_labels, cmap='winter', s=80)
plt.scatter(sk_km.cluster_centers_[:, 0], sk_km.cluster_centers_[:, 1],
            c='red', s=200, marker='X', label='Centroids')
plt.legend()
plt.title('sklearn KMeans — 3 clusters')
plt.show()

---

## Summary

| Concept | Detail |
|---|---|
| `random.sample(range(n), k)` | Pick k unique starting row indices — no repeats |
| `np.dot(a−b, a−b)` | Squared Euclidean distance = Σ(aᵢ−bᵢ)² |
| `distances.index(min(...))` | Which centroid was closest = cluster label |
| `X[cluster_group == type]` | Boolean mask — select all rows of one cluster |
| `.mean(axis=0)` | Column-wise mean → new centroid (one value per feature) |
| `(old == new).all()` | Convergence check — centroids stopped moving |
| Result depends on random init | Run several times or use `init='k-means++'` in sklearn |

### Key limitation
Because centroids are initialised randomly, different runs can give different results.
sklearn's default `init='k-means++'` uses smarter initialisation to avoid this — our scratch version uses pure random sampling.